# RFS-FIM Fetch

Pull flood-extent tiles from the public S3 bucket `floodmap-sandbox` that overlap a
bounding-box AOI, for a chosen return period, then clip/mosaic to a single GeoTIFF.

### Setup

In [44]:
# run once if needed
# %pip install s3fs geopandas rioxarray rasterio gdal

import math
import os
import s3fs
import geopandas as gpd
import numpy as np
import rioxarray
from pathlib import Path
from shapely.geometry import box

# Read tiles straight from the public S3 bucket: no signed credentials, no local download.
os.environ["AWS_NO_SIGN_REQUEST"] = "Yes"
from osgeo import gdal
gdal.UseExceptions()

fs = s3fs.S3FileSystem(anon=True)
BUCKET = "floodmap-sandbox"
DEM = "fabdem"
TIF_NAME = "flows_2,5,10,25,50,100.tif"
RETURN_PERIODS = [2, 5, 10, 25, 50, 100]   # band order in the tif

### 1. Accept AOI

Read the `AOI_bbox` shapefile from the project folder.

In [45]:
AOI_PATH = "Moron.shp"
aoi_name = Path(AOI_PATH).stem

aoi_poly = gpd.read_file(AOI_PATH)
if aoi_poly.crs is None:
    raise ValueError("AOI has no CRS defined.")
if aoi_poly.crs.to_epsg() != 4326:
    aoi_poly = aoi_poly.to_crs(4326)

# Collapse whatever shape we got to its tight axis-aligned bounding box (a rectangle).
minx, miny, maxx, maxy = aoi_poly.total_bounds
aoi = gpd.GeoDataFrame({"name": [aoi_name]},
                       geometry=[box(minx, miny, maxx, maxy)],
                       crs="EPSG:4326")

aoi_bounds = aoi.total_bounds
print(f"AOI '{aoi_name}' bbox (min_lon, min_lat, max_lon, max_lat):")
print(aoi_bounds)

AOI 'Moron' bbox (min_lon, min_lat, max_lon, max_lat):
[-78.65575629  22.07147615 -78.59470568  22.13997427]


### 2. Accept return-period selection

Pick return periods.

In [46]:
SELECTED_RPS = [2, 5, 10, 25, 50, 100]
assert all(rp in RETURN_PERIODS for rp in SELECTED_RPS), f"Pick from {RETURN_PERIODS}"
print("Selected return periods:", SELECTED_RPS)

Selected return periods: [2, 5, 10, 25, 50, 100]


### 3. Derive the 5 variables

From the bbox: floor the minimums and ceiling the maximums to tile boundary extents.

In [47]:
min_lon = math.floor(aoi_bounds[0])
min_lat = math.floor(aoi_bounds[1])
max_lon = math.ceil(aoi_bounds[2])
max_lat = math.ceil(aoi_bounds[3])

print(f"RETURN_PERIOD = {RETURN_PERIODS}")
print(f"lon: {min_lon} -> {max_lon}")
print(f"lat: {min_lat} -> {max_lat}")

RETURN_PERIOD = [2, 5, 10, 25, 50, 100]
lon: -79 -> -78
lat: 22 -> 23


### 4. Build candidate tile keys and keep the ones that exist

Loop every integer lon/lat in the AOI range (SW-corner tiles), build the S3 key,
and keep only tiles that actually exist in the bucket. `range(min, max)` is exclusive
on the top end, which is correct for SW-corner 1 degree tiles.

In [48]:
urls = []
for lon_to_download in range(min_lon, max_lon):
    for lat_to_download in range(min_lat, max_lat):
        key = (f"{BUCKET}/tiles/lon={lon_to_download}/lat={lat_to_download}"
               f"/floodmaps/dem={DEM}/{TIF_NAME}")
        if fs.exists(key):
            urls.append(key)
            print(f"  found: lon={lon_to_download}, lat={lat_to_download}")
        else:
            print(f"  missing: lon={lon_to_download}, lat={lat_to_download}")

print(f"\n{len(urls)} tiles to read from S3.")

  found: lon=-79, lat=22

1 tiles to read from S3.


### 5. Mosaic + clip directly from S3 (gdal.Warp, no download)

Use `gdal.Warp` to read each existing tile over `/vsis3/`, mosaic them, and clip to the
AOI bbox in a single pass. Nothing is copied locally except the final clipped mosaic.
(The per-return-period results are clipped to the true AOI polygon later in step 6.)

In [49]:
os.makedirs("Results", exist_ok=True)

# GDAL reads each tile in place over /vsis3/ (windowed reads, no full download);
# Warp mosaics them and clips to the AOI bbox. Only the result is written to disk.
vsis3_paths = [f"/vsis3/{key}" for key in urls]
minx, miny, maxx, maxy = aoi_bounds
mosaic_path = os.path.join("Results", f"{aoi_name}_mosaic.tif")

gdal.Warp(
    mosaic_path,
    vsis3_paths,
    outputBounds=(minx, miny, maxx, maxy),  # clip to AOI bbox (EPSG:4326)
    dstSRS="EPSG:4326",
    resampleAlg="near",
    multithread=True,
)
print(f"wrote clipped mosaic: {mosaic_path}")

wrote clipped mosaic: Results/Moron_mosaic.tif


### 6. Threshold each return period -> result GeoTIFF

- Open the clipped mosaic and map each return period to its pixel value.
- For each selected return period, build the flood-extent mask and clip to the AOI.
- Write the result locally.

In [50]:
mosaic = rioxarray.open_rasterio(mosaic_path, masked=True).squeeze("band", drop=True)

vals = sorted((int(v) for v in np.unique(mosaic.values[~np.isnan(mosaic.values)]) if v > 0),
              reverse=True)
rp_to_value = dict(zip(sorted(RETURN_PERIODS), vals))
print("return-period -> pixel-value map:", rp_to_value)

written = []
for rp in SELECTED_RPS:
    thresh = rp_to_value[rp]

    mask = ((mosaic > 0) & (mosaic >= thresh)).astype("uint8")
    extent = mask.where(mask == 1, 255).astype("uint8")
    extent = extent.rio.write_crs(mosaic.rio.crs).rio.write_nodata(255)

    # clip to the true AOI polygon (not the bbox) so flood pixels outside it are dropped
    clipped = extent.rio.clip(aoi_poly.geometry.values, aoi_poly.crs, drop=True)
    out_path = os.path.join("Results", f"{aoi_name}_rp{rp}.tif")
    clipped.rio.to_raster(out_path)

    n = int((clipped == 1).sum())
    written.append(out_path)
    print(f"  wrote {out_path}  ({n:,} flooded px in AOI)")

print("\nDone. Files:", written)

return-period -> pixel-value map: {2: 100, 5: 83, 10: 66, 25: 50, 50: 33, 100: 16}
  wrote Results/Moron_rp2.tif  (5,103 flooded px in AOI)
  wrote Results/Moron_rp5.tif  (6,413 flooded px in AOI)
  wrote Results/Moron_rp10.tif  (7,353 flooded px in AOI)
  wrote Results/Moron_rp25.tif  (8,511 flooded px in AOI)
  wrote Results/Moron_rp50.tif  (9,624 flooded px in AOI)
  wrote Results/Moron_rp100.tif  (10,801 flooded px in AOI)

Done. Files: ['Results/Moron_rp2.tif', 'Results/Moron_rp5.tif', 'Results/Moron_rp10.tif', 'Results/Moron_rp25.tif', 'Results/Moron_rp50.tif', 'Results/Moron_rp100.tif']
